# Phase 1: Digital Twin Network Generation
**Objective:** Convert our raw geographical data into a playable, logically sound SUMO simulation network.

We use the bounding box of our downloaded Kigali `.graphml` file to crop the raw OpenStreetMap `.pbf` file. This programmatic execution of `netconvert` automatically handles the logic for guessing traffic lights (`tls.guess`), merging complex junctions, and pruning disconnected roads.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.environment.network_builder import build_sumo_network, validate_network

graphml_path = Path("../data/raw/kigali_network.graphml")
net_output_path = Path("../data/processed/kigali.net.xml")

build_sumo_network(graphml_path=graphml_path, output_net_path=net_output_path)

2026-03-24 10:37:01,218 - INFO - Running netconvert on the locally generated XML...
2026-03-24 10:37:05,859 - INFO - Successfully generated SUMO network at ../data/processed/kigali.net.xml


## Network Validation
To ensure `netconvert` didn't generate a corrupted file or an empty map, we pass the resulting `.net.xml` file through `sumolib` to parse the road network objects.

In [2]:
validate_network(net_path=net_output_path)

2026-03-24 10:37:05,864 - INFO - Validating network topology: ../data/processed/kigali.net.xml
2026-03-24 10:37:07,474 - INFO - Network is valid. Successfully parsed 20230 nodes and 43641 edges.


## 3. Deterministic Background Traffic Generation
To ensure our Reinforcement Learning models are trained and evaluated in a consistent environment, we generate a reproducible baseline of peak-hour traffic. We use a fixed random seed (`42`) to generate 5,000 trips across a 1-hour simulation window.

In [3]:
from src.environment.traffic_generator import generate_background_traffic

rou_output_path = Path("../data/processed/kigali_traffic.rou.xml")

generate_background_traffic(
    net_path=net_output_path, 
    output_rou_path=rou_output_path, 
    num_vehicles=5000, 
    seed=42
)

2026-03-24 10:37:07,482 - INFO - Traffic route file already exists at ../data/processed/kigali_traffic.rou.xml. Skipping generation.
